In [ ]:
# Clone your repository
!git clone https://github.com/ilatims-b/error_measurement.git

# Change working directory to the cloned repo
%cd error_measurement

# Install required packages
!pip install -q sec-edgar-downloader tiktoken openai transformers accelerate

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Retrieve tokens from Kaggle Secrets
user_secrets = UserSecretsClient()

# Hugging Face Login
hf_token = user_secrets.get_secret("...")
login(token=hf_token)

# Groq API Key Setup
try:
    groq_api_key = user_secrets.get_secret("groq")
    os.environ["GROQ_API_KEY"] = groq_api_key
    print("Groq API Key successfully loaded into environment.")
except:
    print("Warning: 'groq' secret not found. You may need to provide it manually in Step 3.")

In [ ]:
!watch -n 3 nvidia-smi

In [ ]:
%%bash
echo "=========================================="
echo " Step 1: Downloading & Processing SEC Data"
echo "=========================================="

python src/dataset.py \
    --email "ce24b119@smail.iitm.ac.in" \
    --download-dir "./data/sec_filings" \
    --output-file "./data/processed_dataset.jsonl" \
    --year 2023

In [ ]:
%%bash
echo "=========================================="
echo " Step 2: Generating Text Continuations"
echo "=========================================="

python src/generate.py \
    --input-file "./data/processed_dataset.jsonl" \
    --output-file "./data/generations.jsonl" \
    --model-name "meta-llama/Llama-3.2-3B-Instruct" \
    --max-new-tokens 512 \
    --num-continuations 3 \
    --temperature 0.7 \
    --top-p 0.9

In [ ]:
%%bash
echo "=========================================="
echo " Step 3: Fact Extraction and Verification"
echo "=========================================="

# Retrieve API key from environment variable (set in Cell 2)
# Fallback to the provided key if secret is missing
API_KEY=${GROQ_API_KEY:-""}

python src/fact_pipeline.py \
    --gen-file "./data/generations.jsonl" \
    --source-file "./data/processed_dataset.jsonl" \
    --output-file "./data/evaluated_generations.jsonl" \
    --api-key "$API_KEY" \
    --base-url "https://api.groq.com/openai/v1" \
    --model-name "llama-3.3-70b-versatile" \
    --chunk-size 128 \
    --extraction_prompt "" \
    --verification_prompt "" \